In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    avg,
    max,
    min,
    lag,
    to_timestamp
)

from pyspark.sql.window import Window

# ==========================================
# CREATE SPARK SESSION
# ==========================================

spark = SparkSession.builder \
    .appName("SensorFeatureEngineering") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")


In [13]:

# ==========================================
# READ CSV FILE
# ==========================================

df = spark.read.csv(
    "sensor_data.csv",
    header=True,
    inferSchema=True
)


In [14]:
df

DataFrame[sensor_id: string, temperature: double, timestamp: timestamp]

In [15]:

# ==========================================
# CONVERT TIMESTAMP
# ==========================================

df = df.withColumn(
    "event_time",
    to_timestamp(col("timestamp"))
)


In [16]:
# ==========================================
# FILTER INVALID DATA
# ==========================================

df = df.filter(
    (col("temperature").isNotNull()) &
    (col("temperature") > 0) &
    (col("temperature") < 400)
)

# ==========================================
# WINDOW FOR EACH SENSOR
# ==========================================

windowSpec = Window.partitionBy("sensor_id") \
                   .orderBy("event_time")

# Recent 3 readings window
recentWindow = windowSpec.rowsBetween(-2, 0)


In [17]:
# ==========================================
# FEATURE 1: PREVIOUS TEMPERATURE
# ==========================================

df = df.withColumn(
    "prev_temp",
    lag("temperature", 1).over(windowSpec)
)

# ==========================================
# FEATURE 2: TEMPERATURE CHANGE RATE
# ΔT = current - previous
# ==========================================

df = df.withColumn(
    "temp_change",
    col("temperature") - col("prev_temp")
)

# ==========================================
# FEATURE 3: MOVING AVERAGE
# ==========================================

df = df.withColumn(
    "moving_avg_temp",
    avg("temperature").over(recentWindow)
)

# ==

In [19]:

# FEATURE 4: MAX TEMPERATURE
# IN RECENT WINDOW
# ==========================================

df = df.withColumn(
    "max_recent_temp",
    max("temperature").over(recentWindow)
)

# ==========================================
# FEATURE 5: MIN TEMPERATURE
# IN RECENT WINDOW
# ==========================================

df = df.withColumn(
    "min_recent_temp",
    min("temperature").over(recentWindow)
)


In [20]:

# ==========================================
# SHOW RESULTS
# ==========================================

df.show(truncate=False)


+---------+-----------+--------------------------+--------------------------+---------+-------------------+------------------+---------------+---------------+
|sensor_id|temperature|timestamp                 |event_time                |prev_temp|temp_change        |moving_avg_temp   |max_recent_temp|min_recent_temp|
+---------+-----------+--------------------------+--------------------------+---------+-------------------+------------------+---------------+---------------+
|SENSOR_1 |93.21      |2026-05-04 17:56:43.704336|2026-05-04 17:56:43.704336|NULL     |NULL               |93.21             |93.21          |93.21          |
|SENSOR_1 |94.92      |2026-05-04 17:58:43.704336|2026-05-04 17:58:43.704336|93.21    |1.710000000000008  |94.065            |94.92          |93.21          |
|SENSOR_1 |99.09      |2026-05-04 18:03:43.704336|2026-05-04 18:03:43.704336|94.92    |4.170000000000002  |95.74000000000001 |99.09          |93.21          |
|SENSOR_1 |41.9       |2026-05-04 18:04:43.704

In [21]:
df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("Spark_output")

print("Feature engineering completed successfully.")

Feature engineering completed successfully.
